In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [ ]:
# try running the following code, if you see errors on Windows OS

# import os
# os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# Hyper-parameters 
hidden_channel_size_1 = 16
hidden_channel_size_2 = 32
num_epochs = 5
batch_size = 100
learning_rate = 0.001

In [4]:
# MNIST dataset 
train_dataset = torchvision.datasets.MNIST(root='data', 
                                           train=True, 
                                           download=True,
                                           transform=transforms.ToTensor())

test_dataset = torchvision.datasets.MNIST(root='data', 
                                          train=False, 
                                          transform=transforms.ToTensor(),
                                          download=True)

# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, 
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset, 
                                          batch_size=batch_size, 
                                          shuffle=False)

In [5]:
len(train_dataset),len(test_dataset)

(60000, 10000)

In [6]:
# CNN
class CNeuralNet(nn.Module):
    def __init__(self,hidden_channel_size_1,hidden_channel_size_2,num_classes=10):
        super(CNeuralNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, hidden_channel_size_1, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(hidden_channel_size_1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(hidden_channel_size_1, hidden_channel_size_2, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(hidden_channel_size_2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc = nn.Linear(7*7*hidden_channel_size_2, num_classes) # originally 28*28, after two MaxPool2d(2), becomes 7*7
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out
    
model = CNeuralNet(hidden_channel_size_1,hidden_channel_size_2).to(device)

In [7]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  

In [8]:
# Train the model
total_step = len(train_loader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))


Epoch [1/5], Step [100/600], Loss: 0.1289
Epoch [1/5], Step [200/600], Loss: 0.1907
Epoch [1/5], Step [300/600], Loss: 0.1378
Epoch [1/5], Step [400/600], Loss: 0.0531
Epoch [1/5], Step [500/600], Loss: 0.0607
Epoch [1/5], Step [600/600], Loss: 0.0431
Epoch [2/5], Step [100/600], Loss: 0.0258
Epoch [2/5], Step [200/600], Loss: 0.0567
Epoch [2/5], Step [300/600], Loss: 0.0324
Epoch [2/5], Step [400/600], Loss: 0.0203
Epoch [2/5], Step [500/600], Loss: 0.0558
Epoch [2/5], Step [600/600], Loss: 0.0300
Epoch [3/5], Step [100/600], Loss: 0.0585
Epoch [3/5], Step [200/600], Loss: 0.0081
Epoch [3/5], Step [300/600], Loss: 0.0275
Epoch [3/5], Step [400/600], Loss: 0.0150
Epoch [3/5], Step [500/600], Loss: 0.0104
Epoch [3/5], Step [600/600], Loss: 0.0936
Epoch [4/5], Step [100/600], Loss: 0.0340
Epoch [4/5], Step [200/600], Loss: 0.0176
Epoch [4/5], Step [300/600], Loss: 0.0513
Epoch [4/5], Step [400/600], Loss: 0.0477
Epoch [4/5], Step [500/600], Loss: 0.0195
Epoch [4/5], Step [600/600], Loss:

In [9]:
# Test the model
model.eval()  # eval mode (batchnorm uses moving mean/variance instead of mini-batch mean/variance)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Test Accuracy of the model on the 10000 test images: {} %'.format(100 * correct / total))

Test Accuracy of the model on the 10000 test images: 98.97 %
